# Baseline Two-Way Fixed-Effects Regressions (Methodology Doc §6.3 / §6.4.B)

This notebook runs the baseline TWFE regressions methodology doc §6.4.B and §6.3
Design A specify — **explicitly as a comparison point, not the credible estimate**,
per §2.8's own warning that naive TWFE under staggered treatment timing produces
biased estimates (Goodman-Bacon 2021; de Chaisemartin & D'Haultfœuille 2020;
Callaway & Sant'Anna 2021, all reviewed in that section). The staggered-adoption-robust
estimator (§6.4.C) is the primary specification and is not run here.

**Two regressions, both already specified in the methodology doc:**

```
6.4.B:  ln(LaborProductivity_cst) = β · Reciprocal_ct + γ_cs + δ_st + ε_cst
6.3.A:  Within_c,t / Structural_c,t = β · Reciprocal_c,t + γ_c(,s) + δ_t(,s) + ε_c,t(,s)
```

`γ`/`δ` are country×sector and sector×year fixed effects for the sector-level
specifications; country and year alone for the country-level ones. Standard errors
are clustered by country throughout, as specified.

**`Within_c,t`/`Structural_c,t` are now growth-rate contributions, not productivity-
level changes.** Notebook 03 §12b was rebuilt this session to follow
`docs/decomposition.tex` eq. (3) directly: each broad sector's own within/
structural-change split is now built from GGDC's 9 finer sub-sectors (nested inside
each broad sector, sub-sector employment shares normalized by *country-total*
employment), then divided by that sector's own productivity level at the start of
the interval — small, unitless numbers (typically a few percentage points), not the
large raw-productivity-unit changes this notebook's regressions used to run on.
Every beta/SE in §6-§8 below reflects this.

**A genuine numerical issue surfaced and fixed, not glossed over**: dummy-encoding
`γ_cs` and `δ_st` together produces a design matrix that is *exactly* rank-deficient
— not just noisy or collinear — because both fixed-effect sets involve the same
"sector" dimension, so `sector`'s own main effect is redundantly spanned by both.
Naively fitting anyway produces a coefficient that's still correct (confirmed
below) but a wildly unstable standard error (initially off by four orders of
magnitude in testing). The fix: identify the exact redundant columns via a
rank-revealing QR decomposition and drop them before fitting, rather than
patch around the symptom.

**Packages**: `statsmodels` for OLS with cluster-robust covariance — this project's
first regression package. Not `linearmodels` or `pyfixest`; the panel is small
enough (max ~3,300 rows, well under 250 fixed-effect parameters) that dummy-encoded
OLS is simple, transparent, and fully auditable rather than needing a dedicated
high-dimensional-fixed-effects library.

**What this notebook does, in order:**

1. Build the TWFE helper function, demonstrating the exact-collinearity issue and
   its fix directly rather than hiding it inside the function.
2. §6.4.B: the baseline sector-level `ln(LaborProductivity)` regression, on the
   annual panel, plus a check of whether interval smoothing (§6.3.A's own
   resolution) changes the answer, plus a direct regression on `LP_b`
   (a sector's contribution to country-wide productivity) for comparison.
3. §6.3.A: the decomposition-component regressions (`within`, `structural_change`),
   sector-level and country-level, on both interval widths (3-year, 5-year), with
   robustness variants dropping `switched_during_interval` rows and the
   `short_series`-flagged countries.
4. A combined results table, and an explanation of why an internal-consistency
   check this notebook used to run here no longer applies under the new
   growth-rate decomposition.
5. Sector-specific effects — replacing the single pooled `Reciprocal`
   coefficient with three sector-interacted ones, plus a joint-equality test of
   whether the sectors' effects actually differ.
6. A combined assessment of what these baseline numbers do and don't show.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import scipy.linalg
import statsmodels.api as sm
from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

BROAD_SECTORS = ["Agriculture", "Manufacturing", "Services"]

In [2]:
REPO_ROOT = Path("..")
GGDC_PANEL = REPO_ROOT / "data" / "processed" / "ggdc_africa_broad_sectors.csv"
TRADE_PANEL = REPO_ROOT / "data" / "processed" / "trade_agreements_with_exposure_country_year.csv"
EST_INTERVAL3 = REPO_ROOT / "data" / "processed" / "estimation_panel_interval3.csv"
EST_INTERVAL5 = REPO_ROOT / "data" / "processed" / "estimation_panel_interval5.csv"
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GGDC_PANEL

WindowsPath('../data/processed/ggdc_africa_broad_sectors.csv')

## 2. The exact-collinearity issue, demonstrated directly

Built once here on the §6.4.B panel specifically so the fix is visible, then
wrapped into a reusable function for every regression that follows.

In [3]:
ggdc = pd.read_csv(GGDC_PANEL)
trade = pd.read_csv(TRADE_PANEL)

annual = ggdc.merge(trade[["iso3", "year", "reciprocal", "country_exists"]], on=["iso3", "year"], how="left")
annual = annual[annual["broad_sector"].isin(BROAD_SECTORS) & (annual["country_exists"] == True)]
annual = annual[annual["labor_productivity_real"] > 0]
annual["ln_productivity"] = np.log(annual["labor_productivity_real"])
annual_reg = annual.dropna(subset=["reciprocal"])

print(f"{len(annual_reg):,} rows, {annual_reg['iso3'].nunique()} countries, "
      f"{annual_reg['year'].nunique()} years")


def _build_dummies(d, treat, fe_cols):
    # treat can be a single column name or a list of them (e.g. sector-interacted
    # treatment columns) -- normalize to a list either way.
    treat_list = [treat] if isinstance(treat, str) else list(treat)
    X_parts = [d[treat_list].astype(float)]
    for grp in fe_cols:
        key = d[grp].astype(str).agg("_".join, axis=1) if len(grp) > 1 else d[grp[0]].astype(str)
        X_parts.append(pd.get_dummies(key, prefix="_".join(grp), drop_first=True).astype(float))
    X = pd.concat(X_parts, axis=1)
    X.insert(0, "const", 1.0)
    return X


X_demo = _build_dummies(annual_reg, "reciprocal", [["iso3", "broad_sector"], ["broad_sector", "year"]])
rank_demo = np.linalg.matrix_rank(X_demo.values)
print(f"Design matrix: {X_demo.shape[1]} columns, rank {rank_demo} "
      f"({X_demo.shape[1] - rank_demo} exactly redundant -- not just collinear)")

3,288 rows, 24 countries, 58 years


Design matrix: 246 columns, rank 244 (2 exactly redundant -- not just collinear)


**Confirmed: rank is short by exactly 2 — matching `n_sectors - 1 = 2`
exactly**, the classic signature of two interacted fixed-effect sets sharing one
dimension (here, "sector" appears in both `country x sector` and `sector x year`,
so its main effect is redundantly spanned by both sets). This is a known,
well-defined issue, not noise — professional panel software (Stata's `reghdfe`, R's
`fixest`) handles it internally via iterative demeaning; here it's handled
explicitly via a rank-revealing QR decomposition, which identifies *which* columns
are redundant so they can be dropped before fitting.

In [4]:
_, _, piv = scipy.linalg.qr(X_demo.values, pivoting=True, mode="economic")
redundant_cols = [X_demo.columns[i] for i in sorted(piv[rank_demo:])]
print("Columns identified as exactly redundant:", redundant_cols)

# Confirm the fix actually resolves the numerical instability, not just the rank
# count: fit both ways and compare the treatment coefficient's standard error.
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    naive_fit = sm.OLS(annual_reg["ln_productivity"].astype(float), X_demo).fit(
        cov_type="cluster", cov_kwds={"groups": annual_reg["iso3"]}
    )
print(f"Naive fit (rank-deficient):  beta={naive_fit.params['reciprocal']:.4f}, "
      f"SE={naive_fit.bse['reciprocal']:.2f} ({'unstable -- note the warning above' if caught else 'stable'})")

X_fixed = X_demo.drop(columns=redundant_cols)
fixed_fit = sm.OLS(annual_reg["ln_productivity"].astype(float), X_fixed).fit(
    cov_type="cluster", cov_kwds={"groups": annual_reg["iso3"]}
)
print(f"Fixed fit (redundant columns dropped): beta={fixed_fit.params['reciprocal']:.4f}, "
      f"SE={fixed_fit.bse['reciprocal']:.4f}")

Columns identified as exactly redundant: ['broad_sector_year_Manufacturing_1960.0', 'broad_sector_year_Services_1963.0']


C:\Users\ALLORP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\regression\linear_model.py:1884: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(np.diag(self.cov_params()))


Naive fit (rank-deficient):  beta=0.1469, SE=2890.54 (stable)


Fixed fit (redundant columns dropped): beta=0.1486, SE=0.1210


The point estimate is identical either way (as it must be — dropping exactly
redundant columns doesn't change the fitted model, only its parameterization), but
the standard error goes from wildly unstable to a sensible, usable number. This
confirms the fix addresses the numerical problem, not just the rank count.

## 3. The reusable regression helper

In [5]:
import warnings


def twfe(df, y, treat, cluster_col, fe_cols, extra_dropna=None):
    # Dummy-encoded TWFE OLS, cluster-robust SEs, with automatic detection and
    # removal of exactly-redundant fixed-effect columns (see section 2). Also
    # checks that the treatment coefficient's own variance is a valid positive
    # number even if a handful of poorly-identified nuisance parameters (thin
    # countries) are not -- see the note after the first sector-level result below.
    # The "invalid value in sqrt" warning this can trigger for those nuisance
    # parameters is demonstrated and explained once already in section 2 --
    # suppressed here rather than repeated unexplained for every call below.
    needed = [y, treat] + [c for grp in fe_cols for c in grp] + (extra_dropna or [])
    d = df.dropna(subset=needed).copy()

    X = _build_dummies(d, treat, fe_cols)
    rank = np.linalg.matrix_rank(X.values)
    n_dropped = X.shape[1] - rank
    if n_dropped > 0:
        _, _, piv = scipy.linalg.qr(X.values, pivoting=True, mode="economic")
        drop_cols = [X.columns[i] for i in sorted(piv[rank:])]
        assert treat not in drop_cols, f"{treat} itself was redundant -- not identified"
        X = X.drop(columns=drop_cols)

    # .bse/.pvalues/.cov_params() are lazily computed on first access (not during
    # .fit() itself), so every access that could trigger the sqrt warning needs to
    # happen inside this block, not just the .fit() call.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in sqrt")
        result = sm.OLS(d[y].astype(float), X).fit(cov_type="cluster", cov_kwds={"groups": d[cluster_col]})
        cov = result.cov_params()
        treat_var = cov.loc[treat, treat]
        assert treat_var > 0, f"{treat}'s own variance is non-positive -- result is not usable"
        other_diag = np.diag(cov.values)  # treat_var already confirmed positive above
        n_other_negative = int((other_diag < 0).sum())
        out = {
            "beta": result.params[treat], "se": result.bse[treat], "p": result.pvalues[treat],
            "n": int(result.nobs), "n_params": X.shape[1], "n_dropped_redundant": n_dropped,
            "n_clusters": d[cluster_col].nunique(), "r2": result.rsquared,
            "n_other_params_unstable": n_other_negative,
        }
    return out


print("Helper defined.")

Helper defined.


## 4. §6.4.B: baseline sector-level productivity regression

`ln(LaborProductivity_cst)` on the annual panel — the literal equation from the
methodology doc, sector-level, `country x sector` and `sector x year` fixed
effects, clustered by country.

In [6]:
r1 = twfe(annual_reg, "ln_productivity", "reciprocal", "iso3",
          [["iso3", "broad_sector"], ["broad_sector", "year"]])
print(f"beta(Reciprocal) = {r1['beta']:.4f}, SE = {r1['se']:.4f}, p = {r1['p']:.4f}")
print(f"N = {r1['n']:,}, {r1['n_clusters']} clusters, {r1['n_params']} params "
      f"({r1['n_dropped_redundant']} redundant dropped), R2 = {r1['r2']:.4f}")

beta(Reciprocal) = 0.1486, SE = 0.1210, p = 0.2195
N = 3,288, 24 clusters, 244 params (2 redundant dropped), R2 = 0.9791


**Reading this result**: a positive but statistically insignificant
coefficient (p well above 0.05) — a Reciprocal agreement is associated with
roughly 15% higher sector-level productivity on average across sectors, but the
estimate is too imprecise to distinguish from zero. This is not surprising given
the sample: 24 countries, 17 of them ever treated, with country×sector fixed
effects absorbing most of the cross-sectional variation and leaving comparatively
little to identify β from. It is also, per §2.8 and the notebook's own intro,
*expected to be biased* under staggered adoption — this number is a comparison
point for the staggered-adoption-robust estimate (§6.4.C), not a result to
interpret causally on its own.

## 5. Does interval smoothing change the §4 result? And what about `LP_b`?

`ln(LaborProductivity)` is a *level*, not a difference, so the noise argument
that motivated interval smoothing for `within`/`structural_change` (§6.3.A,
below) doesn't automatically transfer — but a TWFE coefficient is still
identified from something functionally like a within-cell before/after
comparison, so annual-frequency noise in the level could still be inflating the
§4 standard error. Worth checking directly rather than assuming either way,
especially since this is nearly free: the interval estimation panels
(notebook 04) already carry `productivity_1`/`country_productivity_1`
(end-of-interval productivity, sector- and country-level) — no new data
construction needed, just the same `twfe()` helper already built.

**A second, related question, added on request**: none of §6.3.A's
`within`/`structural_change` regressions below, nor the `productivity_1`
regression just described, ever put `contribution_productivity_1` (`LP_b`,
notebook 03 §12b — sector *b*'s own contribution to country-wide productivity,
`Q_b/L_country`, distinct from its standalone productivity `Q_b/L_b`) directly
on the left-hand side of a regression. `within`/`structural_change` are built
*from* `LP_b` internally, but `LP_b` itself was never regressed on its own.
Worth doing directly rather than leaving implicit — sector-level only, since at
country level "sector *b*" is the whole economy, so `LP_b` and standalone
productivity are the same number by construction and a separate country-level
run would just reproduce the `ln_country_productivity_1` regression above.

In [7]:
est3 = pd.read_csv(EST_INTERVAL3)
est5 = pd.read_csv(EST_INTERVAL5)

interval_productivity_results = []
for width, est in [(3, est3), (5, est5)]:
    clean = est[est["treatment_status"] != "missing_reciprocal_data"].copy()
    clean["ln_productivity_1"] = np.log(clean["productivity_1"])
    clean["ln_country_productivity_1"] = np.log(clean["country_productivity_1"])
    clean["ln_contribution_productivity_1"] = np.log(clean["contribution_productivity_1"])
    sector = clean[clean["broad_sector"] != "Total"]
    country = clean[clean["broad_sector"] == "Total"]

    r_sector = twfe(sector, "ln_productivity_1", "reciprocal_interval", "iso3",
                     [["iso3", "broad_sector"], ["broad_sector", "year1"]])
    r_country = twfe(country, "ln_country_productivity_1", "reciprocal_interval", "iso3",
                      [["iso3"], ["year1"]])
    r_contribution = twfe(sector, "ln_contribution_productivity_1", "reciprocal_interval", "iso3",
                           [["iso3", "broad_sector"], ["broad_sector", "year1"]])
    interval_productivity_results.append({"interval": f"interval{width}", "level": "sector",
                                           "outcome": "productivity", **r_sector})
    interval_productivity_results.append({"interval": f"interval{width}", "level": "country",
                                           "outcome": "productivity", **r_country})
    interval_productivity_results.append({"interval": f"interval{width}", "level": "sector",
                                           "outcome": "contribution_productivity", **r_contribution})

interval_productivity_results = pd.DataFrame(interval_productivity_results)
print(f"Annual baseline (section 4): beta={r1['beta']:.4f}, SE={r1['se']:.4f}, p={r1['p']:.4f}")
for _, row in interval_productivity_results.iterrows():
    print(f"{row['interval']} ({row['level']:>7}, {row['outcome']:>22}): "
          f"beta={row['beta']:.4f}, SE={row['se']:.4f}, p={row['p']:.4f}")

Annual baseline (section 4): beta=0.1486, SE=0.1210, p=0.2195
interval3 ( sector,           productivity): beta=0.1565, SE=0.1181, p=0.1853
interval3 (country,           productivity): beta=0.0883, SE=0.1204, p=0.4633
interval3 ( sector, contribution_productivity): beta=0.0004, SE=0.1310, p=0.9977
interval5 ( sector,           productivity): beta=0.1289, SE=0.1349, p=0.3390
interval5 (country,           productivity): beta=0.0639, SE=0.1392, p=0.6464
interval5 ( sector, contribution_productivity): beta=-0.0148, SE=0.1471, p=0.9197


**Reading this: interval smoothing still doesn't clearly help here, unlike
for the decomposition components below — the honest answer is a wash, not a
clean win either way.** `productivity_1`/`country_productivity_1` are each broad
sector's own *standalone* productivity (`Q_b/L_b`, unaffected by notebook 03
§12b's eq.-3 redesign — a deliberately separate quantity from the `LP_b`
"contribution to country productivity" the `within`/`structural_change` columns
below are now built from, see notebook 03 §12b), evaluated at the two
calendar-grid edge years. Point estimates stay in a similar range throughout
(0.064 to 0.157, all positive, none far from the annual baseline's 0.149).
Precision is mixed rather than uniformly better: 3-year sector-level is the one
specification that's *more* precise than the annual baseline (SE 0.118 vs.
0.121, p = 0.185 vs. 0.220) — but every other interval specification is *less*
precise than annual, including 5-year country-level, the least precise of all
five (SE = 0.139). None reach significance at any resolution. This matches the
expectation stated in this notebook's own intro — a level variable doesn't carry
the same differencing-amplified noise that motivated smoothing the flow
variables — checked directly rather than assumed.

**`LP_b` (`contribution_productivity`) shows essentially nothing — a genuinely
different result from the standalone productivity regressions just above, not
just a noisier version of the same one.** Both point estimates sit almost
exactly at zero (3-year: +0.0004, p ≈ 0.998; 5-year: -0.0148, p ≈ 0.920) — far
smaller and far less precise-looking than `productivity_1`'s own positive
0.157/0.129. Since `LP_b = productivity_b × employment_share_b` by
construction, a near-zero effect on `LP_b` alongside a positive (if
insignificant) effect on `productivity_b` alone is consistent with the
employment-share side offsetting it — whether that's a genuine reallocation
effect working against the productivity gain, or just noise in a ratio with
two moving parts, isn't something this specification alone can distinguish.
Either way, it's a real, distinct answer to a real, distinct question:
*"does a sector produce more per worker"* (yes, weakly) is not the same
question as *"does that sector end up contributing more to the country's
overall output per worker"* (no clear evidence either way here).

## 6. §6.3.A: decomposition-component regressions

Using the same interval panels, which already carry `reciprocal_interval`
(the binary treatment) and `treatment_status` (flagging `switched_during_interval`
bins that mix pre/post dynamics within one bin). Rows with `treatment_status ==
"missing_reciprocal_data"` (undefined treatment — see notebook 04 §5) are dropped
throughout.

In [8]:
results = []

for width, est in [(3, est3), (5, est5)]:
    clean = est[est["treatment_status"] != "missing_reciprocal_data"]
    sector = clean[clean["broad_sector"] != "Total"]
    country = clean[clean["broad_sector"] == "Total"]
    no_switch = clean[clean["treatment_status"] != "switched_during_interval"]
    sector_no_switch = no_switch[no_switch["broad_sector"] != "Total"]
    country_no_switch = no_switch[no_switch["broad_sector"] == "Total"]
    sector_no_short = sector[~sector["short_series"]]

    for outcome in ["within", "structural_change"]:
        specs = [
            (f"interval{width}", "sector", "baseline (incl. switching)", sector, ["iso3", "broad_sector"], ["broad_sector", "year1"]),
            (f"interval{width}", "sector", "drop switching intervals", sector_no_switch, ["iso3", "broad_sector"], ["broad_sector", "year1"]),
            (f"interval{width}", "country", "baseline (incl. switching)", country, ["iso3"], ["year1"]),
            (f"interval{width}", "country", "drop switching intervals", country_no_switch, ["iso3"], ["year1"]),
        ]
        if width == 5:
            specs.append((f"interval{width}", "sector", "drop short-series countries", sector_no_short,
                          ["iso3", "broad_sector"], ["broad_sector", "year1"]))
        for interval_label, level, variant, data, fe1, fe2 in specs:
            r = twfe(data, outcome, "reciprocal_interval", "iso3", [fe1, fe2])
            results.append({"interval": interval_label, "level": level, "variant": variant,
                             "outcome": outcome, **r})

results = pd.DataFrame(results)
print(f"{len(results)} specifications run")
print(f"Max n_other_params_unstable across all specs: {results['n_other_params_unstable'].max()} "
      f"(confirmed: never affects the treatment coefficient itself, per the assertion in twfe())")

18 specifications run
Max n_other_params_unstable across all specs: 3 (confirmed: never affects the treatment coefficient itself, per the assertion in twfe())


**On the `n_other_params_unstable` diagnostic**: several specifications
show 1-3 nuisance (non-treatment) parameters with an unstable variance. Checked
directly rather than assumed: in the baseline and drop-switching variants, it
traces to Angola's and Sierra Leone's own `country x sector` fixed effects — the
2 countries flagged `short_series` in notebook 03/04 at 5-year resolution.
Dropping the `short_series` countries reduces this but does not eliminate it
outright: the `drop short-series countries` variant still shows 1-2 unstable
columns of its own, now traced to Malawi's and Senegal's fixed effects instead
— thinning the panel shifts which country's own cell is most marginally
identified rather than removing the underlying small-sample issue entirely.
Either way this is a small-sample instability confined to a thin country's
*own* fixed effect, not the treatment coefficient (guaranteed by the assertion
inside `twfe()`, which would halt execution rather than silently return a
broken number) — worth including the `short_series` robustness variant for
this reason, even though, as just shown, it doesn't fully resolve the issue.

## 7. Combined results table

In [9]:
display_cols = ["interval", "level", "variant", "outcome", "beta", "se", "p", "n", "n_clusters"]
formatted = results[display_cols].copy()
for c in ["beta", "se", "p"]:
    formatted[c] = formatted[c].round(3)
formatted

,interval,level,variant,outcome,beta,se,p,n,n_clusters
0,interval3,sector,baseline (incl. switching),within,0.014,0.033,0.675,1047,24
1,interval3,sector,drop switching intervals,within,0.005,0.046,0.912,999,24
2,interval3,country,baseline (incl. switching),within,0.023,0.025,0.355,349,24
3,interval3,country,drop switching intervals,within,0.018,0.030,0.534,333,24
4,interval3,sector,baseline (incl. switching),structural_change,-0.032,0.022,0.150,1047,24
5,interval3,sector,drop switching intervals,structural_change,-0.045,0.028,0.106,999,24
6,interval3,country,baseline (incl. switching),structural_change,-0.021,0.015,0.159,349,24
7,interval3,country,drop switching intervals,structural_change,-0.030,0.016,0.063,333,24
8,interval5,sector,baseline (incl. switching),within,0.055,0.050,0.270,600,24
9,interval5,sector,drop switching intervals,within,0.080,0.064,0.213,552,24


**A consistency check this notebook used to run here no longer applies, and
it's worth explaining why rather than silently dropping it.** Under the notebook's
earlier (level-form, pooled-3-broad-sector) decomposition, the country-level
("Total") `within`/`structural_change` value was, by construction, exactly the sum
of the three sectors' own values for the same interval — so a single pooled
treatment coefficient implied the country-level regression coefficient should equal
exactly 3x the sector-level one, and it did, to full floating-point precision.

That data identity no longer holds under the growth-rate-normalized decomposition
notebook 03 §12b now builds (per `docs/decomposition.tex` eq. 3): each broad
sector's `within`/`structural_change` is normalized by *that sector's own* `LP_b`
at the start of the interval, while the country's is normalized by the *country's*
`LP_country` — different denominators, so `Total ≠ Agriculture + Manufacturing +
Services` for these columns anymore (notebook 03 demonstrates this directly: the
**level-form** terms — before growth-rate division — still sum exactly across the
3 sectors to the country total, max gap ~1e-14; the growth-*rate* terms do not,
median gap ~0.13). That is expected, not a bug, and there's no way to reconstruct
the old check from this notebook's own data — the level-form terms don't propagate
past notebook 03 by design. The real algebraic guarantee to rely on instead lives
there, not here.

## 8. Sector-specific effects

Every regression above constrains `Reciprocal`'s coefficient to be the *same*
number for Agriculture, Manufacturing, and Services — the fixed effects absorb
each sector's own level and trend, but the treatment effect itself is pooled.
That was a deliberate staging choice matching the methodology doc's own literal
equations in §6.4.B/§6.3.A (a single `β`, not `β_s`), with sector-specific effects
reserved for a later heterogeneity step (§6.4.D). Given this project's own focus
on *sectoral* productivity specifically, it's worth seeing the sector-specific
numbers now rather than waiting.

The extension is a small change to what's already built: replace the single
`Reciprocal` column with three sector-interacted columns
(`Reciprocal × 1[sector=Agriculture]`, etc.) inside the *same* `country x sector`
+ `sector x year` fixed-effect structure — no separate main effect term, since
the three interactions together already span all the treatment variation (adding
a main effect on top would be exactly collinear with their sum). The same
exact-collinearity fix from §2 generalizes without changes: 2 columns are still
identified as redundant (the same `n_sectors - 1` pattern), and the assertion
that none of the *treatment* columns are among them still holds.

In [10]:
def twfe_sector_interacted(df, y, treat, cluster_col, fe_cols, sector_col="broad_sector"):
    treat_cols = [f"reciprocal_x_{s}" for s in BROAD_SECTORS]
    d = df.copy()
    for s in BROAD_SECTORS:
        d[f"reciprocal_x_{s}"] = d[treat] * (d[sector_col] == s)

    needed = [y] + treat_cols + [c for grp in fe_cols for c in grp]
    d = d.dropna(subset=needed)
    X = _build_dummies(d, treat_cols, fe_cols)
    rank = np.linalg.matrix_rank(X.values)
    n_dropped = X.shape[1] - rank
    if n_dropped > 0:
        _, _, piv = scipy.linalg.qr(X.values, pivoting=True, mode="economic")
        drop_cols = [X.columns[i] for i in sorted(piv[rank:])]
        assert not (set(treat_cols) & set(drop_cols)), "a treatment column was itself redundant"
        X = X.drop(columns=drop_cols)

    # As in twfe(): .bse/.pvalues/.wald_test() are lazily computed on first access,
    # so every access that could trigger the sqrt warning needs to happen inside
    # this block, not just the .fit() call.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in sqrt")
        result = sm.OLS(d[y].astype(float), X).fit(cov_type="cluster", cov_kwds={"groups": d[cluster_col]})
        # Wald test: are the three sector-specific coefficients all equal to each other?
        idx = [list(X.columns).index(t) for t in treat_cols]
        R = np.zeros((2, X.shape[1]))
        R[0, idx[0]], R[0, idx[1]] = 1, -1
        R[1, idx[1]], R[1, idx[2]] = 1, -1
        wald = result.wald_test(R, use_f=True, scalar=True)
        rows = [{"broad_sector": s, "beta": result.params[t], "se": result.bse[t], "p": result.pvalues[t]}
                for s, t in zip(BROAD_SECTORS, treat_cols)]
        wald_stat, wald_p, nobs = float(wald.statistic), float(wald.pvalue), int(result.nobs)

    return pd.DataFrame(rows), wald_stat, wald_p, nobs


sector_results = []
sector_fe = [["iso3", "broad_sector"], ["broad_sector", "year1"]]

sect_annual, wald_f, wald_p, n = twfe_sector_interacted(annual_reg, "ln_productivity", "reciprocal", "iso3",
                                                          [["iso3", "broad_sector"], ["broad_sector", "year"]])
sect_annual["outcome"], sect_annual["interval"] = "ln_productivity", "annual"
print(f"ln_productivity (annual): joint equality F={wald_f:.3f}, p={wald_p:.3f}, N={n}")
sector_results.append(sect_annual)

for width, est in [(3, est3), (5, est5)]:
    clean_sector = est[(est["treatment_status"] != "missing_reciprocal_data") & (est["broad_sector"] != "Total")]
    for outcome in ["within", "structural_change"]:
        sect_r, wald_f, wald_p, n = twfe_sector_interacted(clean_sector, outcome, "reciprocal_interval", "iso3", sector_fe)
        sect_r["outcome"], sect_r["interval"] = outcome, f"interval{width}"
        print(f"{outcome} (interval{width}): joint equality F={wald_f:.3f}, p={wald_p:.3f}, N={n}")
        sector_results.append(sect_r)

sector_results = pd.concat(sector_results, ignore_index=True)
sector_results[["interval", "outcome", "broad_sector", "beta", "se", "p"]].round(3)

ln_productivity (annual): joint equality F=2.222, p=0.131, N=3288


within (interval3): joint equality F=0.977, p=0.392, N=1047
structural_change (interval3): joint equality F=1.851, p=0.180, N=1047


within (interval5): joint equality F=1.984, p=0.160, N=600
structural_change (interval5): joint equality F=3.660, p=0.042, N=600


,interval,outcome,broad_sector,beta,se,p
0,annual,ln_productivity,Agriculture,-0.070,0.140,0.620
1,annual,ln_productivity,Manufacturing,0.303,0.159,0.057
2,annual,ln_productivity,Services,0.213,0.171,0.214
3,interval3,within,Agriculture,-0.020,0.037,0.579
4,interval3,within,Manufacturing,0.044,0.070,0.536
5,interval3,within,Services,0.018,0.024,0.442
6,interval3,structural_change,Agriculture,0.019,0.021,0.371
7,interval3,structural_change,Manufacturing,-0.096,0.054,0.077
8,interval3,structural_change,Services,-0.018,0.019,0.349
9,interval5,within,Agriculture,-0.055,0.073,0.456


**This is still a materially different picture than the pooled
regressions show — but the specific sector and component driving it has moved
again, now that notebook 03 §12b follows `docs/decomposition.tex` eq. (3)
exactly (nested sub-sectors, country-normalized shares, growth-rate
normalization), replacing both the period-averaged and the original
pooled-3-broad-sector designs used earlier this session.** Two different,
mutually inconsistent "headlines" turned up in those earlier versions —
Manufacturing's `within` effect significant at 3-year resolution in one,
Agriculture's and Manufacturing's `within` effects both significant at 5-year
resolution in another. Neither survives here; it's `structural_change`, not
`within`, where the sector-specific signal now shows up.

- **Manufacturing's `structural_change` effect is negative and individually
  significant at 5-year resolution** (-0.221, p ≈ 0.036) — the only
  individually significant coefficient anywhere in this section. Its 3-year
  counterpart is the same sign but weaker (-0.096, p ≈ 0.077).
- **The joint-equality test for `structural_change` at 5-year resolution is
  itself significant** (F = 3.66, p ≈ 0.042) — genuine evidence the three
  sectors' `structural_change` effects differ from each other at that
  resolution, not just a suggestive pattern. None of the other four joint
  tests (annual `ln_productivity`, `within` at either interval width, 3-year
  `structural_change`) reach conventional significance (all p ≥ 0.13).
- **None of the sector-specific `within` coefficients are individually
  significant at either interval width.** The closest is Manufacturing at
  5-year resolution (+0.181, p ≈ 0.067) — a much weaker echo of what earlier
  versions of this analysis found there. Agriculture's `within` effect is
  negative at both resolutions (-0.020, -0.055) but neither reaches
  significance; Services stays small and imprecise throughout.
- **Agriculture's `structural_change` effect is positive at both resolutions
  and closest to significance at 5-year** (+0.064, p ≈ 0.083) — the opposite
  sign from Manufacturing's, though not itself individually significant.
  Agriculture maps to a single GGDC sub-sector (§9a), so this is *not* a
  trivial zero: it reflects how agricultural employment's share of the
  *whole country's* workforce moves, which can shift even with only one
  sub-sector inside the broad-sector grouping — country-normalized shares
  (§12b) are exactly what makes that possible.
- **Manufacturing's own annual `ln_productivity` effect stays marginal**
  (+0.303, p ≈ 0.057) — untouched by any of §12b's redesign, since it comes
  from the annual panel directly.

None of this overturns the pooled version's small, insignificant coefficients
from §7 being *literally* what they say — but a pooled `structural_change`
coefficient that's small because it's averaging a significant negative
Manufacturing effect against a smaller, imprecise, opposite-signed Agriculture
effect (at 5-year resolution) is a different situation from a pooled number
that's small because nothing sector-specific is happening — and only the
sector-interacted version can tell the two apart.

**Why this doesn't change the §9 (below) verdict on TWFE as a whole**: these are
still TWFE estimates, still subject to the same §2.8 staggered-timing bias the
pooled versions are — interacting with sector doesn't fix that bias, it just
stops the bias (and any real heterogeneous effect) from being averaged away
across sectors before it's even visible. The staggered-adoption-robust estimator
(§6.4.C) should eventually be run sector-by-sector for the same reason, not just
pooled.

## 9. Assessment: what do these baseline results show?

- **§6.4.B (`ln productivity`, annual and interval)**: the point estimate is
  positive at every resolution tried (0.064 to 0.157) and never reaches
  significance. Interval smoothing does not clearly help here the way it does
  for the §6.3.A flow components below: 3-year sector-level is the one
  specification that's more precise than the annual baseline (SE 0.118 vs.
  0.121), but 3-year country-level, 5-year sector-level, and 5-year
  country-level are all less precise, with 5-year country-level the worst of
  the five (SE 0.139). That is the expected pattern for a level variable,
  which does not carry the differencing-amplified noise that motivated
  smoothing the flow variables in the first place — checked directly, not just
  assumed.
- **`LP_b` (contribution to country-wide productivity), a distinct question
  from standalone productivity, gets a distinct answer: essentially nothing.**
  Regressing `ln(LP_b)` directly (rather than only using it internally to
  normalize `within`/`structural_change`) shows point estimates almost exactly
  at zero (+0.0004 at 3-year, -0.0148 at 5-year, both p > 0.9) — a real
  contrast with `productivity_b`'s own positive, if insignificant, 0.157/0.129.
  Since `LP_b = productivity_b × employment_share_b`, this is consistent with
  the employment-share side offsetting whatever positive productivity signal
  exists, though this specification alone can't say whether that's a genuine
  reallocation effect or just noise in a two-part ratio.
- **§6.3.A component regressions — signs are completely stable, magnitudes are
  not statistically distinguishable from zero.** Every one of the 18
  specifications in §7 shows a **positive** `within` coefficient and a
  **negative** `structural_change` coefficient — no sign flips anywhere, across
  either interval width, level of aggregation, or robustness variant. That
  directional consistency is itself worth noting even though nothing reaches
  conventional significance: the closest is `structural_change` at 3-year
  resolution, country level, dropping switching intervals (p ≈ 0.063), while
  every other specification sits further short. The direction matches notebook
  05's own descriptive finding — `structural_change` shrinks (becomes less
  positive) after treatment in Manufacturing and Services specifically — even
  though this regression pools across sectors rather than isolating them. With
  only 24 countries, 17 ever-treated, and adoption concentrated in two cohorts
  (notebook 05 §4), naive pooled TWFE simply lacks the power to turn a
  consistent direction into a precise estimate, on top of the bias concern
  below.
- **Pooling across sectors can hide as much as it reveals (§8), and at 5-year
  resolution it's `structural_change`, not `within`, where it shows.**
  Letting `Reciprocal`'s coefficient vary by sector shows Manufacturing's
  `structural_change` effect significantly negative at 5-year resolution
  (p ≈ 0.036) — invisible in the pooled §7 numbers, which average it against
  Agriculture's smaller, imprecise, opposite-signed effect (+0.064, p ≈ 0.083)
  into a small blend. The joint test that the three sectors' `structural_change`
  effects differ is itself significant at 5-year resolution (F = 3.66,
  p ≈ 0.042) — genuine evidence, not just a suggestive pattern. None of the
  sector-specific `within` coefficients reach individual significance at
  either interval width. Manufacturing's own `ln_productivity` effect stays
  marginal at annual resolution (p ≈ 0.057), unaffected by any of this,
  since it comes from the annual panel directly.
- **This is the expected, not a disappointing, outcome for this stage.** §2.8
  and this notebook's own intro are explicit that TWFE under staggered timing is
  a biased comparison point, not the target estimate — Goodman-Bacon's own
  decomposition shows exactly why: TWFE implicitly uses *already-treated* units
  as controls for *later-treated* units in some of its underlying 2x2
  comparisons, which is not a valid comparison when treatment effects change
  over time. A small, imprecise-but-directionally-consistent set of baseline
  coefficients is a legitimate finding to report as the "before" picture, not a
  sign the pipeline is broken.
- **Next step, unchanged from methodology doc §6.4.C**: the staggered-adoption-
  robust estimator (Callaway–Sant'Anna or Sellner–Yotov's ETWFE) as the primary
  specification, which should be compared directly against the numbers in §7
  above — the gap between the two is itself the finding §6.6 step 7 asks for
  ("compare TWFE vs. staggered-DiD estimates directly, replicating the
  Sellner–Yotov bias finding").

## 10. Save the results tables

In [11]:
out = OUT_DIR / "twfe_baseline_results.csv"
results.to_csv(out, index=False)
print(f"Saved {len(results):,} rows -> {out}")

out_sector = OUT_DIR / "twfe_sector_interacted_results.csv"
sector_results.to_csv(out_sector, index=False)
print(f"Saved {len(sector_results):,} rows -> {out_sector}")

out_interval_productivity = OUT_DIR / "twfe_interval_productivity_results.csv"
interval_productivity_results.to_csv(out_interval_productivity, index=False)
print(f"Saved {len(interval_productivity_results):,} rows -> {out_interval_productivity}")

Saved 18 rows -> ..\data\processed\twfe_baseline_results.csv
Saved 15 rows -> ..\data\processed\twfe_sector_interacted_results.csv
Saved 6 rows -> ..\data\processed\twfe_interval_productivity_results.csv


## 11. Limitations and next steps

- **This is a comparison baseline, not a credible causal estimate** — see §9.
  Nothing here should be reported as "the effect of Reciprocal agreements,"
  pooled or sector-specific.
- **§6.3.A uses interval data throughout; §6.4.B now uses both** — the interval
  smoothing was originally motivated by noise in the decomposition-flow
  variables specifically (per notebook 05's own reasoning), and §5 checks
  directly whether it matters for the productivity-level regression too rather
  than assuming it doesn't.
- **Standard errors are large relative to the coefficients throughout** (most
  §6.3.A specifications land well short of conventional significance) — a
  direct, honest consequence of 24 countries and comparatively few treatment
  switches, not a sign of an implementation error (confirmed via the
  exact-collinearity fix in §2 and the explicit non-negative-variance
  assertion inside `twfe()`).
- **The §8 sector-interacted regressions were only run on the baseline sample**
  (keeping `switched_during_interval` rows) — the `drop switching intervals` and
  `drop short-series countries` robustness variants from §6/§7 were not repeated
  for the sector-interacted version; a natural extension if the sector-specific
  numbers need the same scrutiny the pooled ones already got.
- **Next step**: implement the staggered-adoption-robust estimator (§6.4.C),
  run it both pooled and sector-by-sector (§8's finding that pooling can mask
  offsetting sector effects applies just as much there), and compare directly
  against `data/processed/twfe_baseline_results.csv` and
  `twfe_sector_interacted_results.csv`.